# 08 · Grouping & Aggregating

Pandas' `.groupby()` is the direct equivalent of SQL's `GROUP BY`.
This notebook also fixes the bug that was throwing a real `KeyError` in your
original file — explained right at the point it broke, below.

In [ ]:
import os
import pandas as pd

os.chdir(r"C:\Users\purve\OneDrive\Desktop\extra\2019survey")
df = pd.read_csv('survey_results_public.csv', index_col='Respondent')
schema_df = pd.read_csv('survey_results_schema.csv', index_col='Column')
# SQL: LOAD DATA INFILE 'survey_results_public.csv' INTO TABLE survey_results_public ...;
#      -- index_col='Respondent' just tells pandas "use this column as the row label",
#      -- similar to ALTER TABLE survey_results_public ADD PRIMARY KEY (Respondent);


In [ ]:
pd.set_option('display.max_columns', 85)   # display settings only, no SQL equivalent
pd.set_option('display.max_rows', 85)


In [ ]:
df.head()
# SQL: SELECT * FROM survey_results_public LIMIT 5;


In [ ]:
df['ConvertedComp'].head(15)
# SQL: SELECT ConvertedComp FROM survey_results_public LIMIT 15;


In [ ]:
df['ConvertedComp'].median()
# SQL: MySQL has no built-in MEDIAN() aggregate, so it has to be built with a window function:
# SELECT AVG(ConvertedComp) AS median_val FROM (
#     SELECT ConvertedComp,
#            ROW_NUMBER() OVER (ORDER BY ConvertedComp) AS rn,
#            COUNT(*)     OVER ()                       AS cnt
#     FROM survey_results_public
#     WHERE ConvertedComp IS NOT NULL
# ) ranked
# WHERE rn IN (FLOOR((cnt+1)/2), FLOOR((cnt+2)/2));


---
**🐛 Bug fix #1 — `df.median()`**

Your original notebook had this commented out:
```python
# df.median()
```
Uncommenting it throws a `TypeError` on current pandas, because `median()` across the
*whole* DataFrame refuses to silently skip non-numeric columns like `MainBranch`,
`Hobbyist`, etc. Fix: tell pandas to only look at numeric columns.

In [ ]:
df.median(numeric_only=True)
# SQL: SELECT (MEDIAN-style window-function subquery, repeated per numeric column) FROM survey_results_public;
# -- i.e. one subquery like the one above for EVERY numeric column (CompTotal, ConvertedComp, WorkWeekHrs, ...)


In [ ]:
df.describe()
# SQL: no single-statement equivalent — this is like running COUNT/AVG/STD/MIN/MAX
# together for every numeric column at once, e.g. for one column:
# SELECT COUNT(ConvertedComp), AVG(ConvertedComp), STDDEV(ConvertedComp),
#        MIN(ConvertedComp), MAX(ConvertedComp)
# FROM survey_results_public;


In [ ]:
df['ConvertedComp'].count()
# SQL: SELECT COUNT(ConvertedComp) FROM survey_results_public;   -- COUNT ignores NULLs, same as pandas here


In [ ]:
df['Hobbyist']
# SQL: SELECT Hobbyist FROM survey_results_public;


In [ ]:
df['Hobbyist'].value_counts()
# SQL: SELECT Hobbyist, COUNT(*) AS cnt
#      FROM survey_results_public
#      GROUP BY Hobbyist
#      ORDER BY cnt DESC;


In [ ]:
df['SocialMedia']
# SQL: SELECT SocialMedia FROM survey_results_public;


In [ ]:
schema_df.loc['SocialMedia']
# SQL: SELECT * FROM survey_results_schema WHERE Column = 'SocialMedia';


In [ ]:
df['SocialMedia'].value_counts()
# SQL: SELECT SocialMedia, COUNT(*) AS cnt
#      FROM survey_results_public
#      GROUP BY SocialMedia
#      ORDER BY cnt DESC;


In [ ]:
df['SocialMedia'].value_counts(normalize=True)   # gives each group's share as a % of the total
# SQL: SELECT SocialMedia,
#             COUNT(*) / (SELECT COUNT(*) FROM survey_results_public WHERE SocialMedia IS NOT NULL) AS pct
#      FROM survey_results_public
#      GROUP BY SocialMedia
#      ORDER BY pct DESC;


In [ ]:
df['Country'].value_counts()
# SQL: SELECT Country, COUNT(*) AS cnt FROM survey_results_public GROUP BY Country ORDER BY cnt DESC;


In [ ]:
country_grp = df.groupby(('Country'))
# NOTE: the extra parentheses around 'Country' do nothing here — ('Country') is still
# just the plain string 'Country', not a tuple (you'd need a trailing comma, ('Country',),
# to make it a real 1-item tuple). So this line is identical to df.groupby('Country').
# SQL: this on its own doesn't run anything yet — it's like preparing
#      "... FROM survey_results_public GROUP BY Country" without a SELECT/aggregate on top yet.


In [ ]:
country_grp.get_group('India')
# SQL: SELECT * FROM survey_results_public WHERE Country = 'India';


In [ ]:
filt = df['Country'] == 'India'
df.loc[filt]['SocialMedia'].value_counts()
# SQL: SELECT SocialMedia, COUNT(*) AS cnt
#      FROM survey_results_public
#      WHERE Country = 'India'
#      GROUP BY SocialMedia
#      ORDER BY cnt DESC;


In [ ]:
country_grp['SocialMedia'].value_counts(normalize=True).loc['China']
# SQL: SELECT SocialMedia,
#             COUNT(*) / (SELECT COUNT(*) FROM survey_results_public
#                         WHERE Country = 'China' AND SocialMedia IS NOT NULL) AS pct
#      FROM survey_results_public
#      WHERE Country = 'China'
#      GROUP BY SocialMedia
#      ORDER BY pct DESC;


In [ ]:
country_grp['ConvertedComp'].median().loc['Germany']
# SQL: same MEDIAN window-function subquery as before, with WHERE Country = 'Germany' added.


In [ ]:
country_grp['ConvertedComp'].agg(['median', 'mean']).loc['Canada']
# SQL: SELECT AVG(ConvertedComp) AS mean,
#             (MEDIAN window-function subquery) AS median
#      FROM survey_results_public
#      WHERE Country = 'Canada';


In [ ]:
filt = df['Country'] == 'India'
df.loc[filt]['LanguageWorkedWith'].str.contains('Python').sum()
# SQL: SELECT COUNT(*) FROM survey_results_public
#      WHERE Country = 'India' AND LanguageWorkedWith LIKE '%Python%';


In [ ]:
country_grp['LanguageWorkedWith'].apply(lambda x: x.str.contains('Python').sum())
# country_grp['LanguageWorkedWith'] is a GROUPED object, so .str isn't available on it directly —
# that's why the string method runs INSIDE the lambda, once per group, via .apply().
# SQL: SELECT Country, SUM(CASE WHEN LanguageWorkedWith LIKE '%Python%' THEN 1 ELSE 0 END) AS uses_python
#      FROM survey_results_public
#      GROUP BY Country;


---
### Building a "% of respondents who know Python" table per country

In [ ]:
country_respondents = df['Country'].value_counts()
country_respondents
# SQL: SELECT Country, COUNT(*) AS count FROM survey_results_public GROUP BY Country;
# NOTE: on current pandas, value_counts() names its result column 'count' (not 'Country') —
# that naming detail is the root cause of the KeyError fixed below.


In [ ]:
country_uses_python = country_grp['LanguageWorkedWith'].apply(lambda x: x.str.contains('Python').sum())
country_uses_python
# SQL: SELECT Country, SUM(CASE WHEN LanguageWorkedWith LIKE '%Python%' THEN 1 ELSE 0 END) AS uses_python
#      FROM survey_results_public
#      GROUP BY Country;


In [ ]:
python_df = pd.concat([country_respondents, country_uses_python], axis='columns', sort=False)
python_df
# SQL: SELECT r.Country, r.count AS count, p.uses_python AS LanguageWorkedWith
#      FROM (SELECT Country, COUNT(*) AS count FROM survey_results_public GROUP BY Country) r
#      JOIN (SELECT Country, SUM(CASE WHEN LanguageWorkedWith LIKE '%Python%' THEN 1 ELSE 0 END) AS uses_python
#            FROM survey_results_public GROUP BY Country) p
#        ON r.Country = p.Country;


---
**🐛 Bug fix #2 — the `KeyError: 'NumRespondents'`**

Your original code ran:
```python
python_df.rename(columns={'Country': 'NumRespondents', 'LanguageWorkedWith': 'NumKnowsPython'}, inplace=True)
```
This *looks* right, but `python_df` never actually had a column called `'Country'` —
as noted above, `value_counts()` names its result column **`'count'`**, and `'Country'`
is only the *index* name. So half of this rename matches nothing and is silently
ignored: `python_df` keeps its `count` column instead of getting `NumRespondents`.
Then this line:
```python
python_df['PctKnowsPython'] = (python_df['NumKnowsPython']/python_df['NumRespondents']) * 100
```
throws `KeyError: 'NumRespondents'`, because that column was never created.

Fix: rename `'count'` (the real column name), not `'Country'`.

In [ ]:
python_df.rename(columns={'count': 'NumRespondents', 'LanguageWorkedWith': 'NumKnowsPython'}, inplace=True)
# SQL: ALTER TABLE python_df RENAME COLUMN count TO NumRespondents;
#      ALTER TABLE python_df RENAME COLUMN LanguageWorkedWith TO NumKnowsPython;


In [ ]:
python_df
# SQL: SELECT * FROM python_df;


In [ ]:
python_df['PctKnowsPython'] = (python_df['NumKnowsPython'] / python_df['NumRespondents']) * 100
python_df
# SQL: ALTER TABLE python_df ADD COLUMN PctKnowsPython DOUBLE;
#      UPDATE python_df SET PctKnowsPython = (NumKnowsPython / NumRespondents) * 100;


In [ ]:
python_df.sort_values(by='PctKnowsPython', ascending=False, inplace=True)
# SQL: SELECT * FROM python_df ORDER BY PctKnowsPython DESC;


In [ ]:
python_df.head(50)
# SQL: SELECT * FROM python_df ORDER BY PctKnowsPython DESC LIMIT 50;


In [ ]:
python_df.loc['Japan']
# SQL: SELECT * FROM python_df WHERE Country = 'Japan';
